In [19]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta

import json
import numpy as np

In [20]:
base_url = "http://personal-laptop.local:8080/"


In [21]:
def create_url(base_url,first_part_endpoint,second_part_endpoint):
    return base_url + first_part_endpoint + second_part_endpoint
def getData(url):
    response = requests.get(url)

    # Check response status
    if response.status_code == 200:
        
        data = response.json()
        print(f"Received {len(data)} records.")
        return data
    else:
        print(f"Error: {response.status_code}")


def saveDataIntoDataFolder(data,data_file_name):
    script_dir = Path().resolve().parent
    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (data_file_name + ".json")
    with open(file_path, 'w') as file:
        if isinstance(data, pd.DataFrame):
            print("It's a DataFrame")
            if  data.empty:
                print("No data to save.")
            else:
                data.to_json(file_path, orient='records', lines=False)             
                print(f"Data saved to {file_path}")

        else:  
            print("It's NOT a DataFrame.")    
            if not data:
                print("No data to save.")
            else:    
                json.dump(data, file)
                print(f"Data saved to {file_path}")

        
def getDataFromServerThenSaveThemIntoDataFolder(url,data_file_name):
    data = getData(url)
    saveDataIntoDataFolder(data,data_file_name)
    
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [22]:
def getTimeSeries(df_userInputData_smallScale):
    first_row= df_userInputData_smallScale.iloc[0]
    last_row = df_userInputData_smallScale.iloc[-1]
    start_time = first_row["epoch_ms"]
    end_time   = last_row["epoch_ms"] 
    first_part_endpoint = "timeSeriesEndpoints/"
    second_part_endpoint = "getTimeSeriesData?start=" + str(start_time) + '&end=' +str(end_time)
    url = create_url(base_url,first_part_endpoint,second_part_endpoint)
    data = getData(url)
    df_timeSeries = pd.DataFrame(data)

    return df_timeSeries
def tranformTimeSeries(df_timeSeries):
    # 1. Select only the needed column
    df_timeSeriesExperiments = df_timeSeries[['timestamp', 'Id', 'BME680:breathVocEquivalent']].copy()
    # 2. Convert `timestamp` (UTC) to Europe/Athens tz-aware datetime
    df_timeSeriesExperiments['timestamp'] = (
        pd.to_datetime(df_timeSeriesExperiments['timestamp'], utc=True)
          .dt.tz_convert('Europe/Athens')
    )
    # 3. Create the two “Id=…:” columns
    df_timeSeriesExperiments['Id=0:BME680:breathVocEquivalent'] = np.where(
        df_timeSeriesExperiments['Id'] == 0, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
    )
    df_timeSeriesExperiments['Id=1:BME680:breathVocEquivalent'] = np.where(
        df_timeSeriesExperiments['Id'] == 1, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
    )
    df_timeSeriesExperiments['Id=2:BME680:breathVocEquivalent'] = np.where(
        df_timeSeriesExperiments['Id'] == 2, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
    )
    
    # 4. Set timestamp as the row index
    #df_timeSeriesExperiments.set_index('timestamp', inplace=True)
    
    # 5. Drop the old Id+raw measurement columns
    df_timeSeriesExperiments = df_timeSeriesExperiments.drop(
        columns=['Id', 'BME680:breathVocEquivalent']
    )
    return df_timeSeriesExperiments

def getAndTransformTimeSeries(df_userInputData_smallScale):
    df_timeSeries = getTimeSeries(df_userInputData_smallScale)  
    df_timeSeriesExperiments = tranformTimeSeries(df_timeSeries)
    return df_timeSeriesExperiments

In [33]:
def filterUserInputData(dataframe,field,comparison,condition="NaN",oldData = False,experimentStateSpecified =["InsertingSourcePollutant"]):
# Define the condition
    print(field)
    if comparison == "equals":
        condition = ((dataframe[field] == condition) & (dataframe["experimentState"].isin(experimentStateSpecified)))
    if comparison == "contains" and condition != "NaN":
      
        condition = ((dataframe[field].str.contains(condition, na=False) ) & (dataframe["experimentState"].isin(experimentStateSpecified)))    
    if comparison == "not-equals":
        condition = (~(dataframe[field] == condition)& (dataframe["experimentState"].isin(experimentStateSpecified)))     
    if comparison =="NaN":    
        condition = ((dataframe[field].isna()) & (dataframe["experimentState"].isin(experimentStateSpecified)))
    # Get the indices of the rows that match the condition
    matching_indices = dataframe[condition].index
    # Initialize a list to collect the desired indices
    all_indices = []
    # Add previous, current, and next indices (if within bounds)
    for idx in matching_indices:
        if oldData == False:
            if idx - 1 >= 0 and idx + 1 < len(dataframe) :
            
                if (dataframe.iloc[idx-1]["experimentState"]=="StartingExperiment") and  (dataframe.iloc[idx+1]["experimentState"]=="RemovingSourcePollutant"):

                    all_indices.append(idx - 1)
                    all_indices.append(idx)
                    all_indices.append(idx + 1)
                #when we didn't use the start experiment state for all the experiments,only the inserting source and remove source
        elif oldData == True: 
            if idx + 1 < len(dataframe) :  
           
                if dataframe.iloc[idx+1]["experimentState"]=="RemovingSourcePollutant":
                    all_indices.append(idx)
                    all_indices.append(idx + 1) 
            
    # Remove duplicates and preserve order
    # Use dict.fromkeys to maintain order while removing duplicates
    all_indices = list(dict.fromkeys(all_indices))
    
    # Filter the DataFrame
    df_userInputData_filtered= dataframe.loc[all_indices]
    df_userInputData_filtered=df_userInputData_filtered.reset_index(drop = True)
    return df_userInputData_filtered

In [24]:
def specifyUserInputDataWanted(df_userInputData_smallScale,fields,comparisons,conditions,oldData = False,experimentStateSpecified = ["InsertingSourcePollutant"]):
    
    
    for field,comparison,condition in zip(fields,comparisons,conditions):
            df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,field,comparison,condition,oldData,experimentStateSpecified)

    
    df_userInputData_smallScale = df_userInputData_smallScale.reset_index(drop=True) 

    return df_userInputData_smallScale

In [25]:


def takeTheUserInputDataFromParticularRoom(df_userInputData,particularRoom,oldData = False,experimentStateSpecified =  ["InsertingSourcePollutant"]):
    df_userInputData_smallScale = filterUserInputData(df_userInputData,"room","equals",particularRoom,oldData,experimentStateSpecified)

    df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,"item-used","equals","Φαρμακευτικό αλκοόλ 95%",oldDataa,experimentStateSpecified)

    df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,"are-fans-on","NaN",oldData,experimentStateSpecified)

    df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,"are-windows-opened","NaN",oldData,experimentStateSpecified)

    df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,"notes","not-equals","ανεμιστήρας",oldData,experimentStateSpecified)

    df_userInputData_smallScale = df_userInputData_smallScale.reset_index(drop=True) 

    return df_userInputData_smallScale
    
def countPositionOfPollutantSource(df_userInputData_smallScale):
    df_unique_position_count = df_userInputData_smallScale.groupby(['front-wall', 'side-right-wall','back-wall','side-left-wall'], dropna=False).size().reset_index(name='count')
    return df_unique_position_count    
    

In [27]:
first_part_endpoint = "dataAnalysisEndpoints/"
second_part_endpoint = "getAllUserInputsExperimentState"        

url = create_url(base_url,first_part_endpoint,second_part_endpoint)


df_userInputData = getData(url)
df_userInputData =pd.DataFrame(df_userInputData)
if 'details' in df_userInputData.columns:
    details_df = df_userInputData["details"].apply(pd.Series)
    
    # Then join with the original DataFrame (drop the nested 'details' if desired)
    df_userInputData = pd.concat([df_userInputData.drop(columns=["details"]), details_df], axis=1)

if 0 in df_userInputData.columns:
    df_userInputData =df_userInputData.drop(columns=[0])    

# Convert epoch (assumes seconds) to datetime in local time
df_userInputData["epoch_ms"] = df_userInputData["timestamp"].apply(lambda x: x["$date"])
df_userInputData["timestamp_local"] = pd.to_datetime(df_userInputData["epoch_ms"], unit="ms").dt.tz_localize("UTC").dt.tz_convert("Europe/Athens")
df_userInputData.drop(columns = ["timestamp"])
dimensions = ['front-wall','side-right-wall','back-wall','side-left-wall']
for dimension in dimensions:
    df_userInputData[dimension] = pd.to_numeric(df_userInputData[dimension], errors='coerce')


Received 501 records.


In [28]:
df_userInputData

,_id,experimentState,timestamp,userInputCategory,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,pollutant-type,quantity-used,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,epoch_ms,timestamp_local
0,{'$oid': '6849ace703a598ff63a65bf8'},StartingExperiment,{'$date': 1749658855857},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749658855857,2025-06-11 19:20:55.857000+03:00
1,{'$oid': '6849aefb03a598ff63a6667f'},InsertingSourcePollutant,{'$date': 1749659387774},ExperimentState,on,on,on,1.5,φαρμακευτική λοτιόν,,VOC,δύο κουταλιές της σούπας,Σαλόνι,1.7,NaN,NaN,NaN,NaN,1749659387774,2025-06-11 19:29:47.774000+03:00
2,{'$oid': '6849b77503a598ff63a68c90'},RemovingSourcePollutant,{'$date': 1749661557509},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749661557509,2025-06-11 20:05:57.509000+03:00
3,{'$oid': '6849ba8d03a598ff63a69cd3'},InsertingSourcePollutant,{'$date': 1749662349567},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ,,VOC,τρεις κουταλιά της σούπας,Σαλόνι,1.7,NaN,NaN,NaN,NaN,1749662349567,2025-06-11 20:19:09.567000+03:00
4,{'$oid': '6849c2dc03a598ff63a6c299'},RemovingSourcePollutant,{'$date': 1749664476491},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749664476491,2025-06-11 20:54:36.491000+03:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,{'$oid': '68adfc02ac4a03c9fe12c5d2'},InsertingSourcePollutant,{'$date': 1756232706584},ExperimentState,on,NaN,NaN,0.7,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",0.9,NaN,NaN,NaN,NaN,1756232706584,2025-08-26 21:25:06.584000+03:00
497,{'$oid': '68adfdf0ac4a03c9fe12cc82'},RemovingSourcePollutant,{'$date': 1756233200196},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1756233200196,2025-08-26 21:33:20.196000+03:00
498,{'$oid': '68ae0039ac4a03c9fe12d41e'},StartingExperiment,{'$date': 1756233785024},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1756233785024,2025-08-26 21:43:05.024000+03:00
499,{'$oid': '68ae0111ac4a03c9fe12d71e'},InsertingSourcePollutant,{'$date': 1756234001710},ExperimentState,on,NaN,NaN,0.6,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,NaN,0.6,NaN,NaN,1756234001710,2025-08-26 21:46:41.710000+03:00


In [31]:
print(df_userInputData["room"].unique())
saveDataIntoDataFolder(df_userInputData,"All User Input Data")

[nan 'Σαλόνι' 'σαλονι' 'Κρεβατοκάμαρα '
 'Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55'
 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
 'Σοφίτα-Όλοι οι αισθητήρες μαζί' 'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί'
 'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90']
C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
It's a DataFrame
Data saved to C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\All User Input Data.json


In [34]:
df_userInputData = loadDataFromFile("All User Input Data")
oldData = False
experimentStateSpecified =["NoSourcePollutantInserted","InsertingSourcePollutant"]
fields = ["room"]
comparisons = ["contains"]
conditions = ["Όλοι οι αισθητήρες μαζί"]
df_userInputData_smallScale = specifyUserInputDataWanted(df_userInputData,fields,comparisons,conditions,oldData,experimentStateSpecified)


C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Loaded 501 records from C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\All User Input Data.json
room


In [35]:
df_userInputData_smallScale

,_id,experimentState,timestamp,userInputCategory,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,pollutant-type,quantity-used,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,epoch_ms,timestamp_local
0,{'$oid': '68a0b932a1fbc302e14444b5'},StartingExperiment,{'$date': 1755363634147},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1755363634147,2025-08-16 17:00:34.147
1,{'$oid': '68a0b9d5a1fbc302e144468f'},NoSourcePollutantInserted,{'$date': 1755363797903},ExperimentState,on,on,on,NaN,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Σοφίτα-Όλοι οι αισθητήρες μαζί,NaN,NaN,NaN,None,on,1755363797903,2025-08-16 17:03:17.903
2,{'$oid': '688801042b6a61abefaee294'},RemovingSourcePollutant,{'$date': 1755377884000},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1755377884000,2025-08-16 20:58:04.000
3,{'$oid': '68a1e16fcf51922818e73582'},StartingExperiment,{'$date': 1755439471205},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1755439471205,2025-08-17 14:04:31.205
4,{'$oid': '68a1e40420b06e7dc691f360'},NoSourcePollutantInserted,{'$date': 1755440132473},ExperimentState,on,on,on,NaN,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Σοφίτα-Όλοι οι αισθητήρες μαζί,NaN,NaN,NaN,None,on,1755440132473,2025-08-17 14:15:32.473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,{'$oid': '68adfc02ac4a03c9fe12c5d2'},InsertingSourcePollutant,{'$date': 1756232706584},ExperimentState,on,None,None,0.7,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",0.9,NaN,NaN,None,None,1756232706584,2025-08-26 18:25:06.584
59,{'$oid': '68adfdf0ac4a03c9fe12cc82'},RemovingSourcePollutant,{'$date': 1756233200196},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1756233200196,2025-08-26 18:33:20.196
60,{'$oid': '68ae0039ac4a03c9fe12d41e'},StartingExperiment,{'$date': 1756233785024},ExperimentState,None,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,1756233785024,2025-08-26 18:43:05.024
61,{'$oid': '68ae0111ac4a03c9fe12d71e'},InsertingSourcePollutant,{'$date': 1756234001710},ExperimentState,on,None,None,0.6,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,NaN,0.6,None,None,1756234001710,2025-08-26 18:46:41.710


In [36]:


transformSeries = getAndTransformTimeSeries(df_userInputData_smallScale)
saveDataIntoDataFolder(df_userInputData_smallScale,"User:Calibration no source pollutant")
saveDataIntoDataFolder(transformSeries,"Data:Calibration no source pollutant")

Received 135893 records.
C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
It's a DataFrame
Data saved to C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\User:Calibration no source pollutant.json
C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
It's a DataFrame
Data saved to C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\Data:Calibration no source pollutant.json
